# How many failure types are still undiscovered?

This notebook runs the pilot on Colab using every CPU core in parallel. No GPU is needed: choose **Runtime → Change runtime type → CPU** (High-RAM if available).

Run the cells in order. The pilot (3 datasets × 3 models × 5 seeds) takes about 10–20 minutes on 8 cores.

In [3]:
# 1. Get the code and check it
!git clone -q https://github.com/ParitKansal/undiscovered-failures.git
%cd undiscovered-failures
!pip install -q -r requirements.txt
!python -m tests.test_estimators
!python -m tests.test_end_to_end | tail -1
import os; print('CPU cores:', os.cpu_count())

/content/undiscovered-failures/undiscovered-failures
ALL 14 ESTIMATOR CHECKS PASSED
[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done   8 out of   8 | elapsed:    2.9s finished
END-TO-END TEST PASSED
CPU cores: 24


In [4]:
# 2. Optional: save results to Google Drive so they survive a disconnect
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/undiscovered-failures'
os.makedirs(OUT, exist_ok=True)

Mounted at /content/drive


In [ ]:
# 3. Run the pilot in parallel (all cores)
!python experiment.py --datasets adult,bank,covertype --models logreg,forest,boosting --seeds 5 --repeats 30 --out {OUT}/pilot.csv --data_home /content/data

In [ ]:
# 4. Summary tables
!python summarize.py {OUT}/pilot.csv

## Pilot v1: wider design and bootstrap intervals
4 datasets (adds MAGIC) × 3 models × 3 seeds × 2 flip rates (0.6, 0.9) × 3 numbers of planted types (15, 30, 60) = 216 jobs.
Each row now also has bootstrap 95% intervals for Chao1, ACE and the jackknife. About 30–40 minutes on 8 cores (faster with more).

In [ ]:
# 5. Run pilot v1 (all cores)
!git pull -q
!python -m tests.test_estimators
!python experiment.py --datasets adult,bank,covertype,magic --models logreg,forest,boosting --seeds 3 --repeats 30 --flips 0.6,0.9 --k_planted 15,30,60 --boot 200 --out {OUT}/pilot_v1.csv --data_home /content/data

In [ ]:
# 6. Summary tables for v1
!python summarize.py {OUT}/pilot_v1.csv

## Pilot v2: bias-calibrated intervals and the unseen share of errors
Same design as v1. New columns: a bias-calibrated bootstrap (estimate and interval), and the share of all errors that belong to types not yet seen, true versus estimated (1 - sample coverage). A few minutes on 24 cores.

In [5]:
# 7. Run pilot v2 (all cores)
!git pull -q
!python -m tests.test_estimators
!python experiment.py --datasets adult,bank,covertype,magic --models logreg,forest,boosting --seeds 3 --repeats 30 --flips 0.6,0.9 --k_planted 15,30,60 --boot 200 --out {OUT}/pilot_v2.csv --data_home /content/data

ALL 14 ESTIMATOR CHECKS PASSED
216 jobs on 24 CPU cores
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 24 concurrent workers.
[Parallel(n_jobs=-1)]: Done  24 tasks      | elapsed:   51.0s
[Parallel(n_jobs=-1)]: Done 114 tasks      | elapsed:  2.5min
/usr/local/lib/python3.13/dist-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
[Parallel(n_jobs=-1)]: Done 213 out of 216 | elapsed:  3.9min remaining:    3.3s
[Parallel(n_jobs=-1)]: Done 216 out of 216 | elapsed:  3.9min finished
wrote /content/drive/MyDrive/undiscovered-failures/pilot_v2.csv 77760 rows


In [6]:
# 8. Summary tables for v2 (the first table holds the key numbers)
!python summarize.py {OUT}/pilot_v2.csv

== Key numbers ==
                 rel_s_obs  rel_ace  rel_jackknife1  rel_ace_bc  cov_ace_b  cov_ace_p  cov_jack_b  cov_jack_p  unseen_true  unseen_err  new_ratio
types    budget                                                                                                                                  
kmeans10 300        -0.197   -0.121          -0.101      -0.102      0.658      0.708       0.691       0.744        0.021      -0.001      0.431
         1000       -0.113   -0.068          -0.065      -0.054      0.692      0.758       0.706       0.783        0.003       0.000      0.261
         3000       -0.062   -0.011          -0.018      -0.000      0.817      0.844       0.832       0.837        0.001       0.000      0.378
kmeans25 300        -0.317   -0.173          -0.138      -0.154      0.588      0.664       0.616       0.668        0.088       0.001      0.764
         1000       -0.160   -0.100          -0.070      -0.076      0.650      0.738       0.732       0.